# TP2 — Reconstrucción 3D con cámara estéreo

**I308 Visión Artificial · UdeSA · Santiago Luis Groba Alonso**

Reconstrucción 3D de un objeto rígido (buda) a partir de pares de imágenes RGB capturados con la cámara estéreo ELP-USB3D1080P02-H120. Las celdas markdown describen, en pasado, lo que hace cada etapa — sirven tanto de guía para escribir el código como de borrador para el informe.

## 0. Setup

Importamos las librerías que usa todo el notebook (`cv2`, `numpy`, `matplotlib`, `open3d`, `tqdm`, `pickle`) y el módulo local `aruco.py` con las helpers de ChArUco copiadas del ejemplo de cátedra. Definimos los paths del dataset (`datasets/stereo_propio/`), las dimensiones del patrón ChArUco usado en escena (5×7, square 52.6 mm, marker 31.3 mm, dictionary `DICT_6X6_250`), los parámetros de SGBM (numDisparities=128, blockSize=5) y el bounding box en coordenadas de mundo que vamos a usar más adelante para recortar el objeto.

## 1. Calibración estéreo

Usamos la herramienta de cátedra [i308-calib](https://github.com/udesa-vision/i308-calib) sobre nuestro set de calibración (~163 pares de un checkerboard 10×7 de lado 24.2 mm) para obtener los intrínsecos `K_L`, `K_R`, las distorsiones `D_L`, `D_R` y los extrínsecos `R`, `T` que relacionan ambas cámaras. La herramienta deja todo serializado en `stereo_calibration.pkl`, que en esta celda cargamos con `pickle` e inspeccionamos. Verificamos que el baseline observado `||T||` quede cerca de los 60 mm declarados por el fabricante de la ELP-USB3D1080P02-H120.

## 2. Rectificación

A partir de la calibración llamamos `cv2.stereoRectify` para conseguir las matrices de proyección `P1`, `P2` y la matriz de reproyección `Q` que necesitamos en la etapa de profundidad. Con `cv2.initUndistortRectifyMap` precomputamos los mapas que aplican simultáneamente la des-distorsión y la rectificación a cada lado. Definimos una función `rectify(left, right)` que envuelve `cv2.remap` para usarla cómodamente en el loop. Visualizamos un par original vs rectificado con líneas horizontales superpuestas para confirmar que las epipolares quedaron alineadas a las filas — pre-condición para que el matching estéreo funcione como búsqueda 1D.

## 3. Disparidad

Elegimos Semi-Global Block Matching (`cv2.StereoSGBM_create`) por su mejor cobertura en zonas de textura pobre frente al StereoBM clásico, manteniendo un costo de cómputo razonable. Convertimos el par rectificado a escala de grises antes de pasarlo al matcher y dividimos la salida por 16 porque OpenCV codifica la disparidad en fracciones de 1/16 píxel. Los parámetros (`P1`, `P2` proporcionales a `3 · blockSize²`, `uniquenessRatio=10`, `speckleWindowSize=100`, `speckleRange=2`) los ajustamos iterando sobre uno o dos pares representativos y mirando el mapa con un colormap de matplotlib.

## 4. Pose de la cámara respecto al mundo

Para fusionar nubes parciales en un único marco de coordenadas necesitamos saber, para cada par, cómo está orientada la cámara izquierda respecto al mundo. Definimos ese mundo anclando el origen al ChArUco que aparece en cada captura. Creamos el `CharucoBoard` con `aruco.create_charuco_board`, lo detectamos en la imagen izquierda **original** (no la rectificada — los intrínsecos `K_L`, `D_L` describen la imagen sin rectificar) y estimamos la pose con la helper `estimate_camera_pose_with_homography` que resuelve PnP. El resultado (`rvec`, `tvec`) lo convertimos a una matriz homogénea 4×4 `T_wc` con `cv2.Rodrigues`. Para sanity-check proyectamos los ejes del mundo sobre la imagen y los dibujamos en RGB.

## 5. Reconstrucción 3D de un par

Combinando el mapa de disparidad con la matriz `Q`, `cv2.reprojectImageTo3D` nos devuelve un array `(H, W, 3)` con la posición 3D de cada píxel en el frame de la cámara izquierda. Filtramos los píxeles con disparidad ≤ 0 (inválidos) y los puntos fuera del rango razonable de profundidad (100-2000 mm en nuestro setup, suficiente para enmarcar el buda sin traer fondo). Para llevar esos puntos al frame del mundo invertimos `T_wc` analíticamente (`T_cw = [R^T | -R^T·t]`) y aplicamos la transformación homogénea al batch entero. Armamos una `o3d.geometry.PointCloud` con esos puntos y les pegamos los colores del píxel correspondiente en la izquierda rectificada.

## 6. Loop sobre todos los pares + acumulación

Iteramos sobre todos los `(left_i.jpg, right_i.jpg)` del directorio `captures/`, repitiendo para cada par la secuencia rectificar → disparidad → detectar ChArUco → estimar pose → reproyectar a 3D → transformar al mundo. Sobre cada nube parcial aplicamos tres filtros: recorte con un `AxisAlignedBoundingBox` definido en coordenadas de mundo (descarta el plano del patrón y todo lo que esté fuera del bounding box del buda), `remove_statistical_outlier` con 20 vecinos y σ=2 para sacar puntos espurios del matching, y `voxel_down_sample` con voxel de 1 mm para mantener la densidad manejable. Acumulamos sumando las point clouds con `+=`. Llevamos un registro de los pares donde la detección del ChArUco o el solvePnP fallaron y los reportamos al final.

## 7. Exportar `.PLY` y estimar el alto del objeto

Guardamos la nube final con `o3d.io.write_point_cloud(..., write_ascii=False)` para que el archivo quede en binario y pese menos que el formato ASCII de PLY. Para estimar el alto del buda en milímetros calculamos la extensión de la nube sobre el eje vertical del mundo — en nuestro setup ese eje es `Z`, porque el ChArUco está apoyado sobre el plano `XY`. Reportamos también las extensiones sobre `X` e `Y` como sanity-check de que la escala métrica de la reconstrucción tenga sentido, y comparamos el alto estimado contra una medición manual del objeto con regla.